# POMA + Llama SEA-LION v3 8B
Chạy smoke trước, pilot 200; chỉ bật final sau khi duyệt pilot.

In [ ]:
%pip install -q "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "jsonschema>=4.0" rouge-score nltk


In [ ]:
MODEL_ID = "aisingapore/Llama-SEA-LION-v3-8B-IT"
SOURCE_MODE = "dataset"
KAGGLE_REPO_ROOT = "/kaggle/input/poma-repo/POMA"
REPO_URL = "https://github.com/NgDinhKhoi0709/POMA.git"
REVISION = "main"
OUTPUT_ROOT = "/kaggle/working/poma_sea_lion"
RUN_FINAL_543 = False
FINAL_IDS_PATH = None


In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path(KAGGLE_REPO_ROOT)
if SOURCE_MODE == "git":
    repo = Path("/kaggle/working/POMA")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REVISION, REPO_URL, str(repo)], check=True)
assert all((repo / "dataset" / name).exists() for name in ["qas_dev.json", "qas_test.json", "table.json"])
os.environ.update({"POMA_LLM_MODEL": "local/sea-lion-v3-8b-it", "POMA_LOCAL_MODEL_ID": MODEL_ID, "POMA_PROMPT_PROFILE": "compact", "POMA_USE_AGENT_HINTS": "true", "POMA_PARALLEL_WORKERS": "1"})


In [ ]:
command = [sys.executable, "scripts/run_sea_lion_kaggle_eval.py", "--repo-root", str(repo), "--output-root", OUTPUT_ROOT, "--phase", "pilot", "--mode", "both", "--model", "local/sea-lion-v3-8b-it"]
subprocess.run(command, cwd=repo, check=True)
if RUN_FINAL_543:
    assert FINAL_IDS_PATH, "Set FINAL_IDS_PATH only after reviewing the pilot."
    subprocess.run(command[:command.index("pilot")] + ["final", "--final-ids-path", FINAL_IDS_PATH], cwd=repo, check=True)
